# ***Iterative Imputer / MICE Algorithm***




**Full Form:** **MICE** stands for **Multivariate Imputation by Chained Equations**.

Yeh ek advanced multivariate imputation technique hai jisme ek feature ki missing value ko baaki sabhi features ko use karke predict kiya jata hai through regression models.

---

### ***🔍 Missing Data Mechanism (Kab Kaunsa Imputer Kaam Karta Hai?)***

MICE ko effectively use karne se pehle missing data ke nature ko samajhna zaroori hota hai:

#### ***1. MCAR (Missing Completely at Random)***
* **Concept:** Data missing hone ke peeche koi pattern ya relation nahi hota; missing hona pure accident ya random error hota hai.
* **Example:** Sensor ka momentary network drop hona, survey form ka ek page hawa se ud jana.
* **MICE Suitability:** Yaha simple mean/median ya KNN bhi theek kaam karta hai, par MICE bhi apply kiya ja sakta hai.

#### ***2. MAR (Missing at Random)***
* **Concept:** Data missing hone ka relation us feature ke apne behavior se nahi, balki dataset ke **doosre observed features** se hota hai.
* **Example:** Survey me females aksar weight column skip kar deti hain, yaani missingness ka relation gender column se hai, na ki weight ki exact value se.
* **MICE Suitability:** **Best fit!** MICE multivariate relations ko capture karta hai, isliye MAR conditions me MICE sabse behtar results deta hai.

#### ***3. MNAR (Missing Not at Random)***
* **Concept:** Missing value ka relation seedha us feature ki actual value ya kisi hidden reason se hota hai (data purposely withhold ya hide kiya gaya ho).
* **Example:** Bohot zyada income wale log tax survey me intentionally income column fill nahi karte.
* **MICE Suitability:** Yaha correlation ke basis par predict karna mushkil hota hai, isliye domain knowledge ya special modeling required hoti hai.

---

### ***⚖️ Advantages & Disadvantages of Iterative Imputer (MICE)***

#### ***✅ Advantages***
* **High Accuracy:** Multivariate relationships aur feature-to-feature dependencies ko model karke fill karta hai, jisse distribution disturb nahi hota.
* **Preserves Correlation:** Simple mean/median ki tarah columns ke aapas ke variance aur covariance ko suppress nahi karta.

#### ***❌ Disadvantages***
* **Computationally Slow:** Multiple iterations me har missing column ke liye machine learning models (jaise Bayesian Ridge) train hote hain, isliye bade datasets par kafi time leta hai.
* **Production & Memory Overhead:** Deployment ke waqt pipeline ko trained estimators aur configuration track karni padti hai, jisse latency aur memory consumption dono badh jate hain.

## ***Working*** 
**We are implemet MICE aur `MICE` Haam  Input col pe hi perform karte hai*** 

### ***1. Actual Dataset (R&D Spend Sample)***

| Index | R&D Spend | Administration | Marketing Spend | Profit |
| :---: | :---: | :---: | :---: | :---: |
| **21** | 8.0 | 15.0 | 30.0 | 11.0 |
| **37** | 4.0 | 5.0 | 20.0 | 9.0 |
| **2** | 15.0 | 10.0 | 41.0 | 19.0 |
| **14** | 12.0 | 16.0 | 26.0 | 13.0 |
| **44** | 2.0 | 15.0 | 3.0 | 7.0 |

### ***2. Removing the target column***

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.0 | 15.0 | 30.0 |
| **37** | 4.0 | 5.0 | 20.0 |
| **2** | 15.0 | 10.0 | 41.0 |
| **14** | 12.0 | 16.0 | 26.0 |
| **44** | 2.0 | 15.0 | 3.0 |

### ***2. Removing the target column***

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.0 | 15.0 | 30.0 |
| **37** | 4.0 | 5.0 | 20.0 |
| **2** | 15.0 | 10.0 | 41.0 |
| **14** | 12.0 | 16.0 | 26.0 |
| **44** | 2.0 | 15.0 | 3.0 |

### ***Step 1 - Fill all the NaN values with mean of respective cols***

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | 9.25 | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | 11.25 | 26.00 |
| **44** | 2.00 | 15.00 | 29.25 |

### ***Step 2 - Remove all col1 missing values***
Hum **Left-to-Right** move karte hain aur sabse pehle pehle column (**R&D Spend**) ki original missing value ko wapas **NaN** bana dete hain, taaki baaki do features (**Administration** aur **Marketing Spend**) ko use karke ise predict kiya ja sake.

---

#### 📊 ***Table 1: R&D Spend Missing Value Reset (Left)***

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.0 | 15.00 | 30.00 |
| **37** | **NaN** | 5.00 | 20.00 |
| **2** | 15.0 | 10.00 | 41.00 |
| **14** | 12.0 | 11.25 | 26.00 |
| **44** | 2.0 | 15.00 | 29.25 |

---

#### 🎯 ***Training Data split for Model (Right Table):***
Is regression task me **R&D Spend** hamara **Target ($y$)** ban jata hai aur baaki columns **Input Features ($X$)**:

* **$X_{train}$ & $y_{train}$ (Rows: 21, 2, 14, 44):** Jahan R&D Spend ki value already available hai.

| Index | Administration | Marketing Spend | Target ($y$): R&D Spend |
| :---: | :---: | :---: | :---: |
| **21** | 15.00 | 30.00 | 8.0 |
| **2** | 10.00 | 41.00 | 15.0 |
| **14** | 11.25 | 26.00 | 12.0 |
| **44** | 15.00 | 29.25 | 2.0 |

* **$X_{test}$ (Row: 37):** Is row ka use karke model missing R&D value ko predict karega.

| Index | Administration | Marketing Spend |
| :---: | :---: | :---: |
| **37** | 5.00 | 20.00 |

___


### ***Step 3 - Predict the missing values of col1 using other cols***

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | **23.14** | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | 11.25 | 26.00 |
| **44** | 2.00 | 15.00 | 29.25 |

___

### ***Step 4 & 5 - Next Column Process (Administration Column)***

Ab hum agle column (**Administration**) par move karte hain aur same iterative process ko repeat karte hain:

---

#### ***📋 Step 4: Remove all Col 2 Missing Values (Left Table)***
* Jo value pehle round me mean se fill hui thi (Row 14 ki Administration value), use wapas **NaN** bana diya jata hai.
* Ab is NaN value ko predict karne ke liye baaki bache columns (**R&D Spend** aur **Marketing Spend**) ko as input use kiya jata hai.

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | 23.14 | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | **NaN** | 26.00 |
| **44** | 2.00 | 15.00 | 29.25 |

---

#### 🔮 ***Step 5: Predict the Missing Values of Col 2 (Right Table)***
* Regression model train hota hai aur Row 14 ki missing **Administration** value ko predict karke fill kar deta hai (jaise yaha **11.06** aayi hai).

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | 23.14 | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | **11.06** | 26.00 |
| **44** | 2.00 | 15.00 | 29.25 |

---

### 💡 ***Iske Baad Kya Hoga?***
Same process **Column 3 (Marketing Spend)** ke liye bhi repeat hogi (Row 44 ki NaN value wapas reset hogi aur predict ki jayegi). Jab sabhi columns ek baar process ho jayenge, toh ise **1 Iteration** bolte hain. Aise hi multiple rounds (iterations) tab tak chalte hain jab tak values stable nahi ho jaati!

### ***Step 6 & 7 - Column 3 Process (Marketing Spend)***

Ab hum teesre column (**Marketing Spend**) par move karte hain:

---

#### ***📋 Step 6: Remove all Col 3 Missing Values (Left Table)***
* Row 44 ki **Marketing Spend** value ko dobara **NaN** set kar diya jata hai.
* Ab is NaN value ko predict karne ke liye updated columns (**R&D Spend** aur **Administration**) input features ($X$) banenge.

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | 23.14 | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | 11.06 | 26.00 |
| **44** | 2.00 | 15.00 | **NaN** |

---

#### ***🎯 Training Data Split (Middle Table)***
* **Input Features ($X_{train}$):** Rows 21, 37, 2, aur 14 jahan Marketing Spend available hai.
* **Target ($y_{train}$):** Marketing Spend column ki actual values.
* **Predict on ($X_{test}$):** Row 44 ke `[2.00, 15.00]` par prediction run hoga.

| Index | R&D Spend | Administration |
| :---: | :---: | :---: |
| **21** | 8.00 | 15.00 |
| **37** | 23.14 | 5.00 |
| **2** | 15.00 | 10.00 |
| **14** | 12.00 | 11.06 |

---

#### 🔮 ***Step 7: Predict the Missing Values of Col 3 (Right Table)***
* Model Row 44 ki missing value predict karta hai, jo yaha **31.56** aati hai.

| Index | R&D Spend | Administration | Marketing Spend |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | 23.14 | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | 11.06 | 26.00 |
| **44** | 2.00 | 15.00 | **31.56** |

---

### 🏁 ***Iteration 1 Complete!***
Teeno columns (Col 1, Col 2, Col 3) ek baar predict hokar update ho chuke hain. Is pure ek round ko **Iteration 1** kaha jata hai. Yahi process agle round (**Iteration 2**) me fir shuru hogi nayi updated values ke sath.

# 🔄 ***Iterative Imputer (MICE) - Iterations & Convergence Process***

---

## 📍 ***Round 1: Iteration 0 vs Iteration 1***

<table>
<tr>
<th>Iteration 0 (Mean Baseline)</th>
<th>Iteration 1 (First Regression Pass)</th>
<th>Difference (|Iter 1 - Iter 0|)</th>
</tr>
<tr>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | **9.25** | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | **11.25** | 26.00 |
| **44** | 2.00 | 15.00 | **29.25** |

</td>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | **23.14** | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | **11.06** | 26.00 |
| **44** | 2.00 | 15.00 | **31.56** |

</td>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 0.00 | 0.00 | 0.00 |
| **37** | **13.89** | 0.00 | 0.00 |
| **2** | 0.00 | 0.00 | 0.00 |
| **14** | 0.00 | **-0.19** | 0.00 |
| **44** | 0.00 | 0.00 | **2.31** |

</td>
</tr>
</table>

### ***💡 Iteration 1 Ki Theory:***
* **Initial State:** Iteration 0 me missing values ko simply unke column ke **Mean** se fill kiya gaya tha (R&D: 9.25, Admin: 11.25, Mkt: 29.25).
* **Execution:** Iteration 1 me har missing feature ke liye sequential regression models train hue aur nayi values predict huin (R&D: 23.14, Admin: 11.06, Mkt: 31.56).
* **Difference Check:** Difference table Iteration 1 aur Iteration 0 ka difference hai. Yaha values me kafi bada gap hai (jaise R&D me **13.89** ka change), jiska matlab values abhi settle nahi huin, agla iteration run karna padega.

---

## 📍*** Round 2: Iteration 1 vs Iteration 2***

<table>
<tr>
<th>Iteration 1</th>
<th>Iteration 2</th>
<th>Difference (|Iter 2 - Iter 1|)</th>
</tr>
<tr>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | **23.14** | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | **11.06** | 26.00 |
| **44** | 2.00 | 15.00 | **31.56** |

</td>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | **23.78** | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | **11.22** | 26.00 |
| **44** | 2.00 | 15.00 | **31.56** |

</td>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 0.00 | 0.00 | 0.00 |
| **37** | **0.64** | 0.00 | 0.00 |
| **2** | 0.00 | 0.00 | 0.00 |
| **14** | 0.00 | **0.16** | 0.00 |
| **44** | 0.00 | 0.00 | **0.00** |

</td>
</tr>
</table>

### ***💡 Iteration 2 Ki Theory:***
* **Refinement:** Iteration 1 ki better values ko input banakar wapas se regression fit kiya gaya.
* **Convergence Trend:** Nayi values aayi: R&D $\rightarrow$ 23.78, Admin $\rightarrow$ 11.22, Mkt $\rightarrow$ 31.56.
* **Difference Drop:** Is baar difference bohot shrink ho gaya (R&D change: 13.89 se girkar **0.64**, Mkt change: **0.00**). Iska matlab model **stability (convergence)** ke bohot kareeb aa chuka hai.

---

## ***📍 Round 3: Iteration 2 vs Iteration 3***

<table>
<tr>
<th>Iteration 2</th>
<th>Iteration 3</th>
<th>Difference (|Iter 3 - Iter 2|)</th>
</tr>
<tr>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | **23.78** | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | **11.22** | 26.00 |
| **44** | 2.00 | 15.00 | **31.56** |

</td>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 8.00 | 15.00 | 30.00 |
| **37** | **24.57** | 5.00 | 20.00 |
| **2** | 15.00 | 10.00 | 41.00 |
| **14** | 12.00 | **11.37** | 26.00 |
| **44** | 2.00 | 15.00 | **45.53** |

</td>
<td>

| Idx | R&D | Admin | Mkt |
| :---: | :---: | :---: | :---: |
| **21** | 0.00 | 0.00 | 0.00 |
| **37** | **0.79** | 0.00 | 0.00 |
| **2** | 0.00 | 0.00 | 0.00 |
| **14** | 0.00 | **0.15** | 0.00 |
| **44** | 0.00 | 0.00 | **13.97** |

</td>
</tr>
</table>

### 💡 ***Iteration 3 Ki Theory:***
* **Fluctuation / Tuning:** Model ne agla cycle run kiya. R&D (0.79) aur Admin (0.15) lagbhag saturate ho chuke hain, jabki Marketing Spend me model correlation shifts ki wajah se spike aaya (**13.97**).
* **Stopping Criteria (Tol):** Algorithm tab tak chalta hai jab tak consecutive iterations ke beech ka difference defined threshold parameter (`tol`) se kam na ho jaye, ya fir maximum iterations (`max_iter`, default 10) complete na ho jayein.